# 04 — Advanced House Prices

Goal: predict `price_usd` and improve model performance step by step.


## Step 1 — Imports
Import pandas/numpy/matplotlib/seaborn/sklearn here.


In [156]:
# your imports here
import pandas as pd

## Step 2 — Load dataset
Path: `data/advanced_house_prices.csv`


In [157]:
# load the dataset here
prices = pd.read_csv("data/advanced_house_prices.csv")
prices

,property_id,area_m2,bedrooms,bathrooms,floor,total_floors,build_year,neighborhood,property_type,condition,...,has_parking,has_balcony,energy_rating,school_rating,crime_rate,distance_to_center_km,monthly_income_area,listing_month,days_on_market,price_usd
0,HP100431,40.5,0,1.0,21,24,2008,Airport,house,good,...,no,yes,B,5.1,5.9,13.5,1891.0,Mar,56,364020
1,HP101590,120.1,2,1.8,14,16,1998,Suburb,apartment,good,...,no,yes,C,7.4,3.2,9.4,3681.0,Dec,64,582381
2,HP100217,74.3,2,3.1,8,11,2020,Suburb,apartment,fair,...,no,no,C,6.8,3.0,12.4,3219.0,Apr,52,395744
3,HP100002,86.0,1,1.0,10,22,1982,Green Hills,apartment,fair,...,no,no,B,9.5,0.5,8.3,4095.0,Jan,74,546630
4,HP101794,58.9,1,1.7,15,26,1969,Green Hills,apartment,good,...,yes,yes,E,8.6,0.9,6.0,3793.0,May,67,486921
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1813,HP101737,150.2,5,3.5,12,26,2009,Suburb,house,excellent,...,yes,yes,C,7.3,2.2,9.8,3128.0,Nov,55,902278
1814,HP101192,63.0,1,1.0,2,9,2003,Green Hills,apartment,fair,...,no,yes,D,8.4,3.7,6.8,3914.0,Oct,52,487009
1815,HP101209,131.5,4,3.5,10,14,1968,Business Bay,townhouse,new,...,yes,yes,C,7.6,0.8,4.1,4567.0,Jul,50,878415
1816,HP101059,98.3,2,NaN,11,16,1978,Riverside,apartment,good,...,no,no,E,8.6,2.0,5.4,3383.0,Sep,50,482626


## Suggested order
1. Inspect data
2. Clean missing/duplicates
3. Encode categorical columns
4. Baseline model
5. Improve with feature engineering/outlier handling
6. Compare results


In [158]:
prices.shape

(1818, 22)

In [159]:
prices.info()

<class 'pandas.DataFrame'>
RangeIndex: 1818 entries, 0 to 1817
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   property_id            1818 non-null   str    
 1   area_m2                1818 non-null   float64
 2   bedrooms               1818 non-null   int64  
 3   bathrooms              1773 non-null   float64
 4   floor                  1818 non-null   int64  
 5   total_floors           1818 non-null   int64  
 6   build_year             1818 non-null   int64  
 7   neighborhood           1818 non-null   str    
 8   property_type          1818 non-null   str    
 9   condition              1785 non-null   str    
 10  renovated              1800 non-null   str    
 11  has_elevator           1818 non-null   str    
 12  has_parking            1818 non-null   str    
 13  has_balcony            1818 non-null   str    
 14  energy_rating          1745 non-null   str    
 15  school_rating  

In [160]:
prices.describe()

,area_m2,bedrooms,bathrooms,floor,total_floors,build_year,school_rating,crime_rate,distance_to_center_km,monthly_income_area,days_on_market,price_usd
count,1818.000000,1818.000000,1773.000000,1818.000000,1818.000000,1818.000000,1786.000000,1791.000000,1818.000000,1797.000000,1818.000000,1.818000e+03
mean,91.358031,2.217272,2.100620,12.664466,19.690319,1995.400440,6.948040,2.977108,6.728438,2898.303840,55.247525,5.648296e+05
std,36.109526,1.228262,0.882169,7.312256,8.518446,17.707873,1.473675,1.721663,4.391030,907.612676,19.322946,1.857273e+05
min,25.900000,0.000000,1.000000,1.000000,1.000000,1965.000000,2.900000,0.500000,0.200000,700.000000,2.000000,8.588800e+04
25%,65.525000,1.000000,1.300000,6.000000,13.000000,1980.000000,5.800000,1.600000,3.000000,2197.000000,42.000000,4.419135e+05
50%,84.750000,2.000000,2.000000,13.000000,20.000000,1995.000000,7.100000,2.800000,5.800000,2844.000000,55.000000,5.564890e+05
75%,107.900000,3.000000,2.700000,19.000000,26.000000,2011.000000,8.000000,4.200000,10.400000,3646.000000,68.000000,6.643182e+05
max,260.000000,6.000000,4.500000,25.000000,39.000000,2025.000000,10.000000,9.200000,19.000000,5144.000000,124.000000,1.804901e+06


In [161]:
prices.isnull().sum()

property_id               0
area_m2                   0
bedrooms                  0
bathrooms                45
floor                     0
total_floors              0
build_year                0
neighborhood              0
property_type             0
condition                33
renovated                18
has_elevator              0
has_parking               0
has_balcony               0
energy_rating            73
school_rating            32
crime_rate               27
distance_to_center_km     0
monthly_income_area      21
listing_month             0
days_on_market            0
price_usd                 0
dtype: int64

In [162]:
numeric_missing_columns = ["bathrooms", "school_rating", "crime_rate", "monthly_income_area"]
for col in numeric_missing_columns:
    prices[col] = prices[col].fillna(prices[col].median())

numeric_missing_columns

['bathrooms', 'school_rating', 'crime_rate', 'monthly_income_area']

In [163]:
categorical_missing_cols = ["condition", "renovated", "energy_rating"]

for col in categorical_missing_cols:
    prices[col] = prices[col].fillna(prices[col].mode()[0])

In [164]:
prices.isnull().sum()

property_id              0
area_m2                  0
bedrooms                 0
bathrooms                0
floor                    0
total_floors             0
build_year               0
neighborhood             0
property_type            0
condition                0
renovated                0
has_elevator             0
has_parking              0
has_balcony              0
energy_rating            0
school_rating            0
crime_rate               0
distance_to_center_km    0
monthly_income_area      0
listing_month            0
days_on_market           0
price_usd                0
dtype: int64

In [165]:
prices["building_age"] = 2026 - prices["build_year"]
prices["floor_ratio"] = prices["floor"] / prices["total_floors"]

prices = prices.drop(columns=["property_id","build_year"])

In [166]:
prices_model = pd.get_dummies(prices, drop_first=True)
prices_model

,area_m2,bedrooms,bathrooms,floor,total_floors,school_rating,crime_rate,distance_to_center_km,monthly_income_area,days_on_market,...,listing_month_Dec,listing_month_Feb,listing_month_Jan,listing_month_Jul,listing_month_Jun,listing_month_Mar,listing_month_May,listing_month_Nov,listing_month_Oct,listing_month_Sep
0,40.5,0,1.0,21,24,5.1,5.9,13.5,1891.0,56,...,False,False,False,False,False,True,False,False,False,False
1,120.1,2,1.8,14,16,7.4,3.2,9.4,3681.0,64,...,True,False,False,False,False,False,False,False,False,False
2,74.3,2,3.1,8,11,6.8,3.0,12.4,3219.0,52,...,False,False,False,False,False,False,False,False,False,False
3,86.0,1,1.0,10,22,9.5,0.5,8.3,4095.0,74,...,False,False,True,False,False,False,False,False,False,False
4,58.9,1,1.7,15,26,8.6,0.9,6.0,3793.0,67,...,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1813,150.2,5,3.5,12,26,7.3,2.2,9.8,3128.0,55,...,False,False,False,False,False,False,False,True,False,False
1814,63.0,1,1.0,2,9,8.4,3.7,6.8,3914.0,52,...,False,False,False,False,False,False,False,False,True,False
1815,131.5,4,3.5,10,14,7.6,0.8,4.1,4567.0,50,...,False,False,False,True,False,False,False,False,False,False
1816,98.3,2,2.0,11,16,8.6,2.0,5.4,3383.0,50,...,False,False,False,False,False,False,False,False,False,True


In [167]:
X = prices_model.drop(columns=["price_usd"])
y = prices_model["price_usd"]
print(X.shape)
print(y.shape)

(1818, 47)
(1818,)


In [168]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

In [169]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

In [170]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred[:10]

array([ 353382.41859329,  537626.38441803,  571253.31989747,
        668951.3126321 ,  598915.81659271,  552698.9010719 ,
        421654.17346565,  740040.4706521 , 1046546.24745726,
        610269.54323351])

In [171]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R² Score:", r2)

MAE: 39689.025153874274
MSE: 3449273012.78383
RMSE: 58730.51177015087
R² Score: 0.8877609616836029


In [172]:
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

results["Error"] = results["Actual"] - results["Predicted"]
results["Absolute_Error"] = results["Error"].abs()

results.head(10)

,Actual,Predicted,Error,Absolute_Error
1761,302413,3.533824e+05,-50969.418593,50969.418593
990,557028,5.376264e+05,19401.615582,19401.615582
135,533485,5.712533e+05,-37768.319897,37768.319897
408,664235,6.689513e+05,-4716.312632,4716.312632
591,632312,5.989158e+05,33396.183407,33396.183407
1737,532156,5.526989e+05,-20542.901072,20542.901072
289,431518,4.216542e+05,9863.826534,9863.826534
802,754429,7.400405e+05,14388.529348,14388.529348
1550,982723,1.046546e+06,-63823.247457,63823.247457
1178,618256,6.102695e+05,7986.456766,7986.456766


In [173]:
results["Absolute_Error"].describe()

count       364.000000
mean      39689.025154
std       43349.936322
min         549.619988
25%       16035.953966
50%       30478.303917
75%       52257.693576
max      517693.345841
Name: Absolute_Error, dtype: float64

In [174]:
worst_cases = results.sort_values("Absolute_Error", ascending=False).head(10)
worst_cases

,Actual,Predicted,Error,Absolute_Error
1341,1304275,786581.654159,517693.345841,517693.345841
1383,488823,864349.298965,-375526.298965,375526.298965
65,194370,395327.950678,-200957.950678,200957.950678
1058,188065,358800.147742,-170735.147742,170735.147742
497,407281,573235.955015,-165954.955015,165954.955015
1562,438703,588324.232561,-149621.232561,149621.232561
730,434041,575487.015449,-141446.015449,141446.015449
1641,681431,814126.019345,-132695.019345,132695.019345
1755,797380,919797.569456,-122417.569456,122417.569456
1084,801466,681053.944413,120412.055587,120412.055587


In [175]:
original_prices = pd.read_csv("data/advanced_house_prices.csv")
original_prices.loc[worst_cases.index]

,property_id,area_m2,bedrooms,bathrooms,floor,total_floors,build_year,neighborhood,property_type,condition,...,has_parking,has_balcony,energy_rating,school_rating,crime_rate,distance_to_center_km,monthly_income_area,listing_month,days_on_market,price_usd
1341,HP100747,119.1,3,2.3,18,31,1995,Business Bay,apartment,good,...,yes,no,C,8.3,2.3,2.4,3526.0,Apr,41,1304275
1383,HP100092,115.5,2,1.5,7,12,2025,Business Bay,house,good,...,yes,yes,C,9.0,1.2,1.0,4290.0,Feb,41,488823
65,HP101750,48.3,2,2.7,15,25,2015,Industrial,townhouse,new,...,no,no,D,4.3,3.9,14.0,1639.0,Mar,41,194370
1058,HP100672,64.9,2,1.7,5,16,1968,Market,house,fair,...,no,yes,E,6.1,4.9,1.7,2301.0,Sep,52,188065
497,HP100954,95.9,3,3.5,21,33,2022,Market,studio,excellent,...,yes,yes,B,4.6,5.0,2.3,2647.0,Apr,54,407281
1562,HP101220,63.6,1,1.9,11,12,1986,Downtown,apartment,good,...,yes,yes,A,6.7,2.6,2.6,4068.0,Jun,80,438703
730,HP100102,97.7,2,1.8,15,15,1992,Suburb,townhouse,excellent,...,no,yes,D,7.3,2.0,11.0,1709.0,Oct,32,434041
1641,HP101222,116.6,3,1.6,9,21,2004,Green Hills,house,fair,...,yes,no,B,9.1,3.7,5.5,3940.0,Oct,38,681431
1755,HP100268,171.3,4,3.9,11,11,1981,Downtown,apartment,good,...,yes,yes,B,7.7,1.2,1.2,NaN,Jul,33,797380
1084,HP100135,94.4,2,1.6,13,26,2016,Downtown,apartment,NaN,...,yes,yes,D,6.4,4.4,0.2,3785.0,May,64,801466


In [176]:
Q1 = prices["price_usd"].quantile(0.25)
Q3 = prices["price_usd"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

lower_bound, upper_bound

(np.float64(108306.375), np.float64(997925.375))

In [177]:
prices_clean = prices[
    (prices["price_usd"] >= lower_bound) &
    (prices["price_usd"] <= upper_bound)
]

prices_clean.shape

(1780, 22)

In [178]:
prices_clean_model = pd.get_dummies(prices_clean, drop_first=True)

X = prices_clean_model.drop(columns=["price_usd"])
y = prices_clean_model["price_usd"]

In [179]:
X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [180]:
model_clean = LinearRegression()
model_clean.fit(X_train_clean, y_train_clean)

y_pred_clean = model_clean.predict(X_test_clean)

In [181]:
mae_clean = mean_absolute_error(y_test_clean, y_pred_clean)
mse_clean = mean_squared_error(y_test_clean, y_pred_clean)
rmse_clean = mse_clean ** 0.5
r2_clean = r2_score(y_test_clean, y_pred_clean)

print("MAE:", mae_clean)
print("RMSE:", rmse_clean)
print("R² Score:", r2_clean)

MAE: 35572.75627920431
RMSE: 48875.53646335121
R² Score: 0.9127532492357678


In [182]:
from sklearn.linear_model import Ridge
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_clean, y_train_clean)

ridge_pred = ridge_model.predict(X_test_clean)

In [183]:
ridge_mae = mean_absolute_error(y_test_clean, ridge_pred)
ridge_mse = mean_squared_error(y_test_clean, ridge_pred)
ridge_rmse = ridge_mse ** 0.5
ridge_r2 = r2_score(y_test_clean, ridge_pred)

print("Ridge MAE:", ridge_mae)
print("Ridge RMSE:", ridge_rmse)
print("Ridge R²:", ridge_r2)

Ridge MAE: 35080.52318271621
Ridge RMSE: 48476.841680883954
Ridge R²: 0.9141708479306171
